# Transformer y mapas de atención

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_attention-mechanisms-and-transformers/transformer.ipynb` · [Lección original](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# La arquitectura Transformer
<a id="sec_transformer"></a>

En [Referencia subsec_cnn-rnn-self-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#subsec-cnn-rnn-self-attention) comparamos CNN, RNN y autoatención. La autoatención permite calcular simultáneamente las representaciones de distintas posiciones y conectar directamente dos posiciones del contexto. Estas propiedades motivan construir arquitecturas profundas basadas en atención.

Algunos modelos anteriores combinaban atención con representaciones recurrentes [Cheng.Dong.Lapata.2016,Lin.Feng.Santos.ea.2017,Paulus.Xiong.Socher.2017](https://d2l.ai/chapter_references/zreferences.html). El Transformer prescinde de capas recurrentes y convolucionales [Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html), pero no consta solo de atención: incluye redes feed-forward por posición, conexiones residuales y normalización. Aunque se propuso inicialmente para tareas secuencia a secuencia de texto, su estructura se ha extendido a lenguaje, visión, voz y otros dominios.


In [ ]:
import math
import pandas as pd
import torch
from torch import nn
from laboratorio import d2l

## Modelo
Como ejemplo de la arquitectura encoder–decoder, la arquitectura general del transformador se presenta en [Referencia fig_transformer](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html#fig-transformer). Como podemos ver, el transformador se compone de un codificador y un decodificador. En contraste con la atención de Bahdanau para el aprendizaje secuencia-secuencia en [Referencia fig_s2s_attention_details](https://d2l.ai/chapter_attention-mechanisms-and-transformers/bahdanau-attention.html#fig-s2s-attention-details), las embeddings de secuencia de entrada (fuente) y salida (objetivo) se agregan con codificación posicional antes de ser introducidos en el codificador y el decodificador que apila módulos basados en la autoatención.

![Arquitectura Transformer.](../recursos/originales/transformer.svg)

<a id="fig_transformer"></a>

Ahora proporcionamos una visión general de la arquitectura del transformador en [Referencia fig_transformer](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html#fig-transformer). En un nivel alto, el codificador del transformador es una pila de múltiples capas idénticas, donde cada capa tiene dos subcapas (o bien se denota como $\textrm{sublayer}$). La primera es una agrupación de autoatención multicabeza y la segunda es una red feed-forward aplicada por posición. Específicamente, en la autoatención del codificador, las consultas, las claves y los valores son todos de las salidas de la capa de codificador anterior. Inspirado en el diseño de ResNet de [Referencia sec_resnet](https://d2l.ai/chapter_convolutional-modern/resnet.html#sec-resnet), se utiliza una conexión residual alrededor de ambas capas. En el transformador, para cualquier entrada $\mathbf{x} \in \mathbb{R}^d$ en cualquier posición de la secuencia, requerimos que $\textrm{sublayer}(\mathbf{x}) \in \mathbb{R}^d$ para que la conexión residual $\mathbf{x} + \textrm{sublayer}(\mathbf{x}) \in \mathbb{R}^d$ sea factible. Esta adición de la conexión residual es inmediatamente seguida por la normalización de la capa [Ba.Kiros.Hinton.2016](https://d2l.ai/chapter_references/zreferences.html). Como resultado, el codificador del transformador emite una representación vector $d$-dimensional para cada posición de la secuencia de entrada.

El decodificador Transformer también es una pila de múltiples capas idénticas con conexiones residuales y normalización de capas. Además de las dos subcapas descritas en el codificador, el decodificador inserta una tercera subcapa, conocida como el atención cruzada encoder–decoder, entre estas dos. En el atención cruzada encoder–decoder, las consultas son de las salidas de la subcapa de autoatención del decodificador, y las claves y los valores son de las salidas del codificador. En la autoatención decodificador, las consultas, las claves y los valores son todos de las salidas de la capa decodificador anterior. Sin embargo, cada posición en el decodificador sólo se permite atender a todas las posiciones en el decodificador hasta esa posición.

Ya hemos descrito e implementado la atención multi-cabeza basada en productos de punto escalado en [Referencia sec_multihead-attention](https://d2l.ai/chapter_attention-mechanisms-and-transformers/multihead-attention.html#sec-multihead-attention) y la codificación posicional en [Referencia subsec_positional-encoding](https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#subsec-positional-encoding). En lo siguiente, vamos a implementar el resto del modelo Transformer.

## Redes feed-forward por posición
<a id="subsec_positionwise-ffn"></a>

La red feed-forward aplicada por posición transforma la representación en todas las posiciones de secuencia usando el mismo MLP. Por eso la llamamos * positionwise*. En la implementación de abajo, la entrada `X` con forma (tamaño de lote, número de pasos de tiempo o longitud de secuencia en tokens, número de unidades ocultas o dimensión de característica) será transformada por un MLP de dos capas en un tensor de salida de forma (tamaño de lote, número de pasos de tiempo, `ffn_num_outputs`).


In [ ]:
class PositionWiseFFN(nn.Module):  #@save
    """La red de avance hacia la posición."""
    def __init__(self, ffn_num_hiddens, ffn_num_outputs):
        super().__init__()
        self.dense1 = nn.LazyLinear(ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.LazyLinear(ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

El siguiente ejemplo muestra que **la dimensión más interna de un tensor cambia** al número de salidas en la red de avance de posición. Puesto que el mismo MLP se transforma en todas las posiciones, cuando las entradas en todas estas posiciones son las mismas, sus salidas también son idénticas.


In [ ]:
ffn = PositionWiseFFN(4, 8)
ffn.eval()
ffn(torch.ones((2, 3, 4)))[0]

## Conexión residual y normalización de capas
Ahora vamos a centrarnos en el componente "añadir & norma" en [Referencia fig_transformer](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html#fig-transformer). Como describimos al principio de esta sección, esta es una conexión residual inmediatamente seguida por la normalización de capas. Ambos son clave para las arquitecturas profundas efectivas.

En [Referencia sec_batch_norm](https://d2l.ai/chapter_convolutional-modern/batch-norm.html#sec-batch-norm), explicamos cómo la normalización por lotes (BatchNorm) se hace más reciente y redimensiona a través de los ejemplos dentro de un minibatch. Como se discutió en [Referencia subsec_layer-normalization-in-bn](https://d2l.ai/chapter_convolutional-modern/batch-norm.html#subsec-layer-normalization-in-bn), la normalización por lotes (BatchNorm) es lo mismo que la normalización por lotes (BatchNorm), excepto que la primera se normaliza a través de la dimensión de características, disfrutando así de beneficios de independencia de escala e independencia de tamaño de lotes. A pesar de sus aplicaciones generalizadas en la visión por computadora, la normalización por lotes (BatchNorm) suele ser empíricamente menos efectiva que la normalización por lotes (BatchNorm) en tareas de procesamiento de lenguaje natural, donde las entradas a menudo son secuencias de longitud variable.

El siguiente fragmento de código **compara la normalización a través de diferentes dimensiones por normalización de capas y normalización por lotes (BatchNorm)**.


In [ ]:
ln = nn.LayerNorm(2)
bn = nn.LazyBatchNorm1d()
X = torch.tensor([[1, 2], [2, 3]], dtype=torch.float32)
# Calcular la media y la varianza de X en el modo de entrenamiento
print('layer norm:', ln(X), '\nbatch norm:', bn(X))

Ahora podemos implementar la clase `AddNorm` **usando una conexión residual seguida de normalización de capas**. Abandono también se aplica para regularización.


In [ ]:
class AddNorm(nn.Module):  #@save
    """La conexión residual seguida de normalización de la capa."""
    def __init__(self, norm_shape, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(norm_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

La conexión residual requiere que las dos entradas sean de la misma forma de modo que ** el tensor de salida también tenga la misma forma después de la operación de adición**.


In [ ]:
add_norm = AddNorm(4, 0.5)
shape = (2, 3, 4)
d2l.check_shape(add_norm(torch.ones(shape), torch.ones(shape)), shape)

## Codificador
<a id="subsec_transformer-encoder"></a>

Con todos los componentes esenciales para ensamblar el codificador Transformer, vamos a empezar por implementar ** una sola capa dentro del codificador**. La siguiente clase `TransformerEncoderBlock` contiene dos subcapas: autoatención multi-cabeza y redes de avance hacia la posición, donde se emplea una conexión residual seguida de normalización de la capa alrededor de ambas subcapas.


In [ ]:
class TransformerEncoderBlock(nn.Module):  #@save
    """El bloque de codificador Transformer."""
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout,
                 use_bias=False):
        super().__init__()
        self.attention = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                dropout, use_bias)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(num_hiddens, dropout)

    def forward(self, X, valid_lens):
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        return self.addnorm2(Y, self.ffn(Y))

Como podemos ver, ** ninguna capa en el codificador del transformador cambia la forma de su entrada.**


In [ ]:
X = torch.ones((2, 100, 24))
valid_lens = torch.tensor([3, 2])
encoder_blk = TransformerEncoderBlock(24, 48, 8, 0.5)
encoder_blk.eval()
d2l.check_shape(encoder_blk(X, valid_lens), X.shape)

En la siguiente **Encoder Transformer** implementación, apilamos instancias `num_blks` de las clases `TransformerEncoderBlock` anteriores. Desde que usamos la codificación posicional fija cuyos valores están siempre entre $-1$ y $1$, multiplicamos los valores de las embeddings de entrada aprendebles por la raíz cuadrada de la dimensión de incrustación para volver a escalar antes de resumir la incrustación de entrada y la codificación posicional.


In [ ]:
class TransformerEncoder(d2l.Encoder):  #@save
    """El codificador del transformador."""
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens,
                 num_heads, num_blks, dropout, use_bias=False):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = d2l.PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module("block"+str(i), TransformerEncoderBlock(
                num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias))

    def forward(self, X, valid_lens):
        # Puesto que los valores de codificación posicional están entre -1 y 1, la incrustación
        # los valores se multiplican por la raíz cuadrada de la dimensión de inserción
        # para reescalar antes de que se resuman
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(X, valid_lens)
            self.attention_weights[
                i] = blk.attention.attention.attention_weights
        return X

A continuación especificamos hiperparametros para **crear un codificador de dos capas**. La forma de la salida del codificador de transformadores es (tamaño del lote, número de pasos de tiempo, `num_hiddens`).


In [ ]:
encoder = TransformerEncoder(200, 24, 48, 8, 2, 0.5)
d2l.check_shape(encoder(torch.ones((2, 100), dtype=torch.long), valid_lens),
                (2, 100, 24))

## Decodificador
Como se muestra en [Referencia fig_transformer](https://d2l.ai/chapter_attention-mechanisms-and-transformers/transformer.html#fig-transformer), ** el decodificador de transformadores se compone de múltiples capas idénticas**. Cada capa se implementa en la siguiente clase `TransformerDecoderBlock`, que contiene tres subcapas: autoatención de decodificadores, encodificadores--atención de decodificadores y redes de avance de avance de posición. Estas subcapas emplean una conexión residual a su alrededor seguida de normalización de capas.

Como describimos anteriormente en esta sección, en la autoatención de decodificadores multicabeza enmascarada (la primera subcapa), las consultas, las claves y los valores provienen de las salidas de la capa decodificadora anterior. Cuando se entrenan modelos secuencia-secuencia, se conocen tokens en todas las posiciones (pasos de tiempo) de la secuencia de salida. Sin embargo, durante la predicción, la secuencia de salida se genera token por token; por lo tanto, en cualquier paso de tiempo decodificador sólo se pueden utilizar los tokens generados en la autoatención decodificadora. Para preservar la autorregresión en el decodificador, su autoatención enmascarada especifica `dec_valid_lens` de modo que cualquier consulta solo atienda a todas las posiciones en el decodificador hasta la posición de consulta.


### Nota docente de Hespérides

La atención no distingue por sí sola el orden de una permutación; la información posicional introduce esa estructura. Varias cabezas permiten combinar relaciones en subespacios distintos. Cada bloque reúne atención, red por posición, normalización y conexiones residuales. Distingue la atención propia del encoder, la propia causal del decoder y la atención cruzada hacia el encoder. El notebook traduce secuencias con un Transformer encoder–decoder; no es un LLM preentrenado ni reproduce su escala.

Vínculo con los apuntes: sesión 5, «Transformer y mapas de atención».


In [ ]:
class TransformerDecoderBlock(nn.Module):
    # El bloque I en el decodificador Transformer
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout, i):
        super().__init__()
        self.i = i
        self.attention1 = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                 dropout)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.attention2 = d2l.MultiHeadAttention(num_hiddens, num_heads,
                                                 dropout)
        self.addnorm2 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(ffn_num_hiddens, num_hiddens)
        self.addnorm3 = AddNorm(num_hiddens, dropout)

    def forward(self, X, state):
        enc_outputs, enc_valid_lens = state[0], state[1]
        # Durante el entrenamiento, se procesan todos los tokens de cualquier secuencia de salida
        # al mismo tiempo, así que state[2][self.i] no es Ninguno como inicializado.
        # decodificando cualquier token de secuencia de salida por token durante la predicción,
        # state[2][self.i] contiene representaciones de la salida decodificada en
        # el bloque i-th hasta el paso de tiempo actual
        if state[2][self.i] is None:
            key_values = X
        else:
            key_values = torch.cat((state[2][self.i], X), dim=1)
        state[2][self.i] = key_values
        if self.training:
            batch_size, num_steps, _ = X.shape
            # Forma de dec_valid_lens: (batch_size, num_steps), donde cada
            # fila es [1, 2, ..., num_steps]
            dec_valid_lens = torch.arange(
                1, num_steps + 1, device=X.device).repeat(batch_size, 1)
        else:
            dec_valid_lens = None
        # Autoatención
        X2 = self.attention1(X, key_values, key_values, dec_valid_lens)
        Y = self.addnorm1(X, X2)
        # Atención al codificador-decodificador. Forma de enc_outputs:
        # (tamaño_batch, num_steps, num_hiddens)
        Y2 = self.attention2(Y, enc_outputs, enc_outputs, enc_valid_lens)
        Z = self.addnorm2(Y, Y2)
        return self.addnorm3(Z, self.ffn(Z)), state

Para facilitar operaciones de producto escalar escalado en el codificador--decodificador de atención y operaciones de adición en las conexiones residuales, **la dimensión de característica (`num_hiddens`) del decodificador es la misma que la del codificador.**


In [ ]:
decoder_blk = TransformerDecoderBlock(24, 48, 8, 0.5, 0)
X = torch.ones((2, 100, 24))
state = [encoder_blk(X, valid_lens), valid_lens, [None]]
d2l.check_shape(decoder_blk(X, state)[0], X.shape)

Ahora **construimos todo el decodificador del transformador** compuesto de instancias `num_blks` de `TransformerDecoderBlock`. Al final, una capa totalmente conectada calcula la predicción para todos los tokens de salida posibles `vocab_size`. Tanto de los pesos de autoatención del decodificador como del encodificador--pesos de atención del decodificador se almacenan para su visualización posterior.


In [ ]:
class TransformerDecoder(d2l.AttentionDecoder):
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens, num_heads,
                 num_blks, dropout):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.num_blks = num_blks
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = d2l.PositionalEncoding(num_hiddens, dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module("block"+str(i), TransformerDecoderBlock(
                num_hiddens, ffn_num_hiddens, num_heads, dropout, i))
        self.dense = nn.LazyLinear(vocab_size)

    def init_state(self, enc_outputs, enc_valid_lens):
        return [enc_outputs, enc_valid_lens, [None] * self.num_blks]

    def forward(self, X, state):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        self._attention_weights = [[None] * len(self.blks) for _ in range (2)]
        for i, blk in enumerate(self.blks):
            X, state = blk(X, state)
            # Pesos de autoatención de decodificador
            self._attention_weights[0][
                i] = blk.attention1.attention.attention_weights
            # Pesos de la atención del codificador-decodificador
            self._attention_weights[1][
                i] = blk.attention2.attention.attention_weights
        return self.dense(X), state

    @property
    def attention_weights(self):
        return self._attention_weights

## Entrenamiento

Aquí especificamos que tanto el codificador del transformador como el decodificador del transformador tienen dos capas usando la atención de 4 cabezas. Como en [Referencia sec_seq2seq_training](https://d2l.ai/chapter_recurrent-modern/seq2seq.html#sec-seq2seq-training), entrenamos el modelo del transformador para el aprendizaje secuencia-secuencia en el conjunto de datos de traducción automática en inglés--francés.


In [ ]:
data = d2l.MTFraEng(batch_size=128)
num_hiddens, num_blks, dropout = 256, 2, 0.2
ffn_num_hiddens, num_heads = 64, 4
encoder = TransformerEncoder(
    len(data.src_vocab), num_hiddens, ffn_num_hiddens, num_heads,
    num_blks, dropout)
decoder = TransformerDecoder(
    len(data.tgt_vocab), num_hiddens, ffn_num_hiddens, num_heads,
    num_blks, dropout)
model = d2l.Seq2Seq(encoder, decoder, tgt_pad=data.tgt_vocab['<pad>'],
                    lr=0.001)
trainer = d2l.Trainer(max_epochs=30, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

Después del entrenamiento, usamos el modelo Transformer para **translatar unas pocas frases en inglés** al francés y calcular sus puntuaciones de BLEU.


In [ ]:
engs = ['go .', 'i lost .', 'he\'s calm .', 'i\'m home .']
fras = ['va !', 'j\'ai perdu .', 'il est calme .', 'je suis chez moi .']
preds, _ = model.predict_step(
    data.build(engs, fras), d2l.try_gpu(), data.num_steps)
for en, fr, p in zip(engs, fras, preds):
    translation = []
    for token in data.tgt_vocab.to_tokens(p):
        if token == '<eos>':
            break
        translation.append(token)
    print(f'{en} => {translation}, bleu,'
          f'{d2l.bleu(" ".join(translation), fr, k=2):.3f}')

La forma de los pesos de autoatención del codificador es (número de capas de codificador, número de cabezas de atención, `num_steps` o número de consultas, `num_steps` o número de pares de valores clave).


In [ ]:
_, dec_attention_weights = model.predict_step(
    data.build([engs[-1]], [fras[-1]]), d2l.try_gpu(), data.num_steps, True)
enc_attention_weights = torch.cat(model.encoder.attention_weights, 0)
shape = (num_blks, num_heads, -1, data.num_steps)
enc_attention_weights = enc_attention_weights.reshape(shape)
d2l.check_shape(enc_attention_weights,
                (num_blks, num_heads, data.num_steps, data.num_steps))

En la autoatención del codificador, ambas consultas y claves provienen de la misma secuencia de entrada. Dado que los tokens de relleno no tienen significado, con la longitud válida especificada de la secuencia de entrada ninguna consulta atiende a las posiciones de tokens de relleno. En lo siguiente, dos capas de pesos de atención multi-cabeza se presentan fila por fila. Cada cabeza asiste independientemente basado en un subespacio de representación separado de consultas, claves y valores.


In [ ]:
d2l.show_heatmaps(
    enc_attention_weights.cpu(), xlabel='Posiciones de claves',
    ylabel='Posiciones de consultas', titles=['Cabeza %d' % i for i in range(1, 5)],
    figsize=(7, 3.5))

**Para visualizar los pesos de autoatención decodificadores y los pesos de atención decodificadores, necesitamos más manipulaciones de datos.** Por ejemplo, llenamos los pesos de atención enmascarados con cero. Tenga en cuenta que los pesos de autoatención decodificadores y los pesos de atención de decodificadores tienen las mismas preguntas: el símbolo de inicio de secuencia seguido por los tokens de salida y, posiblemente, los tokens de fin de secuencia.


In [ ]:
dec_attention_weights_2d = [head[0].tolist()
                            for step in dec_attention_weights
                            for attn in step for blk in attn for head in blk]
dec_attention_weights_filled = torch.tensor(
    pd.DataFrame(dec_attention_weights_2d).fillna(0.0).values)
shape = (-1, 2, num_blks, num_heads, data.num_steps)
dec_attention_weights = dec_attention_weights_filled.reshape(shape)
dec_self_attention_weights, dec_inter_attention_weights = \
    dec_attention_weights.permute(1, 2, 3, 0, 4)

In [ ]:
d2l.check_shape(dec_self_attention_weights,
                (num_blks, num_heads, data.num_steps, data.num_steps))
d2l.check_shape(dec_inter_attention_weights,
                (num_blks, num_heads, data.num_steps, data.num_steps))

Debido a la propiedad autorregresiva de la autoatención del decodificador, ninguna consulta atiende a pares de clave--valor después de la posición de consulta.


In [ ]:
d2l.show_heatmaps(
    dec_self_attention_weights[:, :, :, :],
    xlabel='Posiciones de claves', ylabel='Posiciones de consultas',
    titles=['Cabeza %d' % i for i in range(1, 5)], figsize=(7, 3.5))

Similar al caso en la autoatención del codificador, a través de la longitud válida especificada de la secuencia de entrada, ** ninguna consulta de la secuencia de salida atiende a esos tokens de relleno de la secuencia de entrada.**


In [ ]:
d2l.show_heatmaps(
    dec_inter_attention_weights, xlabel='Posiciones de claves',
    ylabel='Posiciones de consultas', titles=['Cabeza %d' % i for i in range(1, 5)],
    figsize=(7, 3.5))

Aunque la arquitectura Transformer fue originalmente propuesta para el aprendizaje secuencial, como descubriremos más adelante en el libro, ya sea el codificador Transformer o el decodificador Transformer se utiliza a menudo individualmente para diferentes tareas de aprendizaje profundo.

## Resumen
El transformador es una instancia del codificador--decodificador arquitectura, aunque el codificador o el decodificador se pueden utilizar individualmente en la práctica. En la arquitectura del transformador, la autoatención multi-cabeza se utiliza para representar la secuencia de entrada y la secuencia de salida, aunque el decodificador tiene que preservar la propiedad autorregresiva a través de una versión enmascarada. Tanto las conexiones residuales como la normalización de capas en el transformador son importantes para entrenar un modelo muy profundo. La red feed-forward aplicada por posición en el modelo del transformador transforma la representación en todas las posiciones de secuencia usando el mismo MLP.

## Ejercicios
1. Entrena un transformador más profundo en los experimentos. ¿Cómo afecta la velocidad de entrenamiento y el rendimiento de la traducción?
1. ¿Es una buena idea reemplazar la atención por producto escalar escalado con atención aditiva en el transformador?
1. Para el modelado del lenguaje, ¿deberíamos usar el codificador Transformer, decodificador, o ambos? ¿Cómo diseñaría este método?
1. ¿Qué desafíos pueden enfrentar los Transformers si las secuencias de entrada son muy largas?
1. ¿Cómo mejoraría la eficiencia computacional y de memoria de Transformers? Consejo: puede referirse al artículo de la encuesta de [Tay.Dehghani.Bahri.ea.2020](https://d2l.ai/chapter_references/zreferences.html).


[Debate del original](https://discuss.d2l.ai/t/1066)
